# Blackjack Reinforcement Learning Experiment
In this notebook, we train a Reinforcement Learning agent using **Tabular Q-Learning** to play Blackjack. The agent interacts with the custom environment defined in the `blackjack.py` file.

## The Deck & The Rules

* The game uses a standard French playing card deck (or multiple combined decks in extended mode).
* Number cards count as their face value, face cards count as 10, and Aces can count as either 11 or 1.
* **Objective:** Achieve a higher score than the dealer without exceeding 21 points.

## Import Fail Troubleshooting

In [ ]:
# import os
# os.chdir(r'C:\Users\freese\Desktop\Allgemein\GitLab\ml-blackjack')

## 1. Setup & Hyperparameters
First, we import the necessary libraries as well as our custom environment from the `blackjack.py` file. We also define the hyperparameters required for the Q-Learning algorithm.

In [ ]:
import numpy as np
import collections
import random
from blackjack import BlackjackEnv
from tqdm import tqdm

# Hyperparameter
alpha = 0.05
gamma = 0.95
num_episodes = 500000

# Q-Tables
Q_basic = collections.defaultdict(lambda: np.zeros(2))
Q_extended = collections.defaultdict(lambda: np.zeros(2))

In [ ]:
def choose_action(state, Q, epsilon):
    if random.random() < epsilon:
        return random.choice([0, 1])
    return np.argmax(Q[state])

In [ ]:
def train_agent(env, Q, num_episodes, alpha, gamma):
    epsilon = 1.0
    epsilon_min = 0.05
    epsilon_decay = 0.999995

    for episode in tqdm(range(num_episodes), desc="Training Progress"):
        state = env.reset()
        done = False

        while not done:
            action = choose_action(state, Q, epsilon)
            next_state, reward, done = env.step(action)

            best_next = np.argmax(Q[next_state])
            td_target = reward + gamma * Q[next_state][best_next] * (not done)
            Q[state][action] += alpha * (td_target - Q[state][action])

            state = next_state

        epsilon = max(epsilon_min, epsilon * epsilon_decay)

    return Q

In [ ]:
from tqdm import tqdm

def evaluate_agent(env, Q, episodes=10000):
    wins = losses = draws = 0

    for _ in tqdm(range(episodes), desc="Evaluating Agent"):
        state = env.reset()
        done = False

        while not done:
            action = np.argmax(Q[state])
            state, reward, done = env.step(action)

        if reward == 1:
            wins += 1
        elif reward == -1:
            losses += 1
        else:
            draws += 1

    winrate = wins / episodes
    avg_reward = (wins - losses) / episodes

    return winrate, avg_reward, wins, losses, draws

In [ ]:
def print_results(name, results):
    winrate, avg_reward, wins, losses, draws = results
    
    print(f"\n{name}")
    print(f"Wins: {wins} | Losses: {losses} | Draws: {draws}")
    print(f"Winrate: {winrate*100:.2f}%")
    print(f"Avg Reward: {avg_reward:.4f}")

## 2. The Training Loop (Q-Learning)
We let the agent play through the specified number of episodes. After each action, we update the Q-table based on the received reward using the Bellman optimality equation:

$$Q(s, a) \leftarrow Q(s, a) + \alpha \left( r + \gamma \max_{a'} Q(s', a') - Q(s, a) \right)$$

In [ ]:
env_basic = BlackjackEnv(state_mode="basic")
env_extended = BlackjackEnv(state_mode="extended")


print("Training BASIC Agent...")
Q_basic = train_agent(env_basic, Q_basic, num_episodes, alpha, gamma)

print("\nTraining COUNTING Agent...")
Q_extended = train_agent(env_extended, Q_extended, num_episodes, alpha, gamma)


## 3. Evaluating the Trained Agent
To evaluate how well the learned strategy performs, we test the agent over 10,000 separate rounds. Here, exploration is completely disabled (equivalent to $\epsilon = 0$), so the agent acts strictly based on the highest expected utility in its Q-table.

In [ ]:

print("\nEvaluating BASIC Agent...")
basic_results = evaluate_agent(BlackjackEnv("basic"), Q_basic)

print("\nEvaluating COUNTING Agent...")
extended_results = evaluate_agent(BlackjackEnv("extended"), Q_extended)



print_results("BASIC AGENT", basic_results)
print_results("COUNTING AGENT", extended_results)
